# ⚙️ Notebook 2: Feature Engineering & Preprocessing
## AI-Driven Credit Risk Evaluation
**Author:** Bandaru Yashwanth | B.Sc. Actuarial Science, Amity University Noida

---
### 🎯 Objective
Transform raw data into ML-ready features. Inspired by actuarial risk principles used in insurance and credit underwriting.

### 📋 Steps
1. Handle missing values
2. Engineer new actuarial-style risk features
3. Encode categorical variables
4. Scale numerical features
5. Train/test split
6. Save processed data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import pickle
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
print('✅ Libraries loaded')

In [ ]:
df = pd.read_csv('../data/credit_risk_raw.csv')
print(f'Raw data shape: {df.shape}')
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum()>0]}')

## 1️⃣ Handle Missing Values

In [ ]:
# Median imputation for numerical columns with missing values
# Why median? Robust to outliers — important for financial data
num_imputer = SimpleImputer(strategy='median')
cols_to_impute = ['credit_score', 'employment_years', 'num_credit_lines']

df[cols_to_impute] = num_imputer.fit_transform(df[cols_to_impute])

print(f'✅ Missing values after imputation: {df.isnull().sum().sum()}')
print('Strategy: Median imputation (robust to outliers in financial data)')

## 2️⃣ Actuarial Feature Engineering
As an Actuarial Science student, I designed features that mirror real-world underwriting ratios used by banks and insurance companies.

In [ ]:
# ── ACTUARIAL RISK RATIOS ──────────────────────────────────────────

# 1. Debt-to-Income Ratio (DTI) — most important underwriting metric
#    Banks reject loans if DTI > 43% (RBI guideline)
df['debt_to_income_ratio'] = (df['existing_debt'] / df['annual_income']).round(4)

# 2. Loan-to-Income Ratio — affordability measure
df['loan_to_income_ratio'] = (df['loan_amount'] / df['annual_income']).round(4)

# 3. Monthly EMI Burden (approximate)
df['monthly_emi_burden'] = (df['loan_amount'] / df['loan_term_months']).round(2)

# 4. EMI as % of monthly income
df['emi_to_monthly_income'] = (df['monthly_emi_burden'] / (df['annual_income'] / 12)).round(4)

# 5. Credit Score Tier (actuarial classification)
df['credit_tier'] = pd.cut(df['credit_score'],
                            bins=[0, 550, 620, 680, 750, 900],
                            labels=['Very Poor', 'Poor', 'Fair', 'Good', 'Excellent'])

# 6. Income Tier
df['income_tier'] = pd.cut(df['annual_income'],
                            bins=[0, 25000, 50000, 100000, 200001],
                            labels=['Low', 'Lower-Mid', 'Upper-Mid', 'High'])

# 7. Age Risk Group (actuarial age banding)
df['age_group'] = pd.cut(df['age'],
                          bins=[0, 25, 35, 45, 55, 100],
                          labels=['Young Adult', 'Early Career', 'Mid Career', 'Senior', 'Pre-Retirement'])

# 8. Financial Stress Score (composite feature)
df['financial_stress'] = (
    df['debt_to_income_ratio'] * 0.4 +
    df['loan_to_income_ratio'] * 0.3 +
    df['emi_to_monthly_income'] * 0.3
).round(4)

print('✅ New features engineered:')
new_feats = ['debt_to_income_ratio', 'loan_to_income_ratio', 'monthly_emi_burden',
             'emi_to_monthly_income', 'credit_tier', 'income_tier', 'age_group', 'financial_stress']
for f in new_feats:
    print(f'  • {f}')

In [ ]:
# Visualise the engineered features vs default
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# DTI vs Default
df.boxplot(column='debt_to_income_ratio', by='default', ax=axes[0],
           boxprops=dict(color='#2C3E50'), medianprops=dict(color='red', linewidth=2))
axes[0].set_title('Debt-to-Income Ratio vs Default', fontweight='bold')
axes[0].set_xticklabels(['No Default', 'Default'])
axes[0].set_xlabel('')

# Credit Tier Default Rate
ct_dr = df.groupby('credit_tier', observed=True)['default'].mean() * 100
colors = ['#E74C3C', '#E67E22', '#F1C40F', '#2ECC71', '#27AE60']
axes[1].bar(ct_dr.index, ct_dr.values, color=colors[:len(ct_dr)], edgecolor='black')
for i, v in enumerate(ct_dr.values):
    axes[1].text(i, v+0.3, f'{v:.1f}%', ha='center', fontweight='bold')
axes[1].set_title('Default Rate by Credit Tier', fontweight='bold')
axes[1].set_ylabel('Default Rate (%)')
axes[1].tick_params(axis='x', rotation=20)

# Financial Stress vs Default
df.boxplot(column='financial_stress', by='default', ax=axes[2],
           boxprops=dict(color='#2C3E50'), medianprops=dict(color='red', linewidth=2))
axes[2].set_title('Financial Stress Score vs Default', fontweight='bold')
axes[2].set_xticklabels(['No Default', 'Default'])
axes[2].set_xlabel('')

plt.suptitle('Engineered Features vs Default', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/engineered_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 3️⃣ Encode Categorical Variables

In [ ]:
df_encoded = df.copy()

# Nominal categories → One-Hot Encoding (no ordinal relationship)
nominal_cols = ['employment_type', 'education', 'property_ownership', 'loan_purpose']
df_encoded = pd.get_dummies(df_encoded, columns=nominal_cols, drop_first=True)

# Ordinal categories → Label Encoding (order matters)
ordinal_mappings = {
    'credit_tier':  {'Very Poor': 1, 'Poor': 2, 'Fair': 3, 'Good': 4, 'Excellent': 5},
    'income_tier':  {'Low': 1, 'Lower-Mid': 2, 'Upper-Mid': 3, 'High': 4},
    'age_group':    {'Young Adult': 1, 'Early Career': 2, 'Mid Career': 3, 'Senior': 4, 'Pre-Retirement': 5}
}
for col, mapping in ordinal_mappings.items():
    df_encoded[col] = df_encoded[col].map(mapping)

# Drop customer_id (not a feature)
df_encoded = df_encoded.drop(columns=['customer_id'])

print(f'Shape after encoding: {df_encoded.shape}')
print(f'Total features: {df_encoded.shape[1] - 1} (excluding target)')
print(f'\nAll columns: {list(df_encoded.columns)}')

## 4️⃣ Train/Test Split & Feature Scaling

In [ ]:
X = df_encoded.drop(columns=['default'])
y = df_encoded['default']

# 80/20 split with stratification to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train set: {X_train.shape[0]:,} samples  |  Default rate: {y_train.mean()*100:.1f}%')
print(f'Test set:  {X_test.shape[0]:,} samples  |  Default rate: {y_test.mean()*100:.1f}%')
print('✅ Stratified split ensures same class distribution in both sets')

In [ ]:
# Scale features — important for Logistic Regression and comparison models
# NOTE: fit ONLY on training data, then transform both sets
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_test_scaled  = scaler.transform(X_test)         # ONLY transform on test (no leakage!)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X.columns)

print('✅ Scaling complete')
print('⚠️  Key rule: NEVER fit_transform on test data — that causes data leakage!')

In [ ]:
# Save processed data
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
X_train_scaled.to_csv('../data/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../data/X_test_scaled.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

# Save scaler for deployment
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('✅ All processed data saved to ../data/')
print('✅ Scaler saved to ../models/scaler.pkl')
print('\n🚀 Ready for Notebook 3: Model Training!')